In [2]:
import pandas as pd
import numpy as np

# 1. Creating directory structure
for folder in ['data', 'visuals']:
    if not os.path.exists(folder):
        os.makedirs(folder)
        
np.random.seed(55)
n_subs = 10000

data = pd.DataFrame({
    'user_id': range(1, n_subs + 1),
    'tenure_months': np.random.randint(1, 24, n_subs),
    'monthly_charges': np.random.uniform(9.99, 19.99, n_subs),
    'total_hours_watched': np.random.gamma(shape=2, scale=20, size=n_subs),
    'customer_support_calls': np.random.poisson(1, n_subs),
    'payment_method': np.random.choice(['Credit Card', 'PayPal', 'Mailed Check'], n_subs)
})

# Logic for Churn (Target Variable)
# Users watch less, have more support calls, and use "Mailed Check" are more likely to churn
logit = (0.05 * data['customer_support_calls'] + 
         -0.02 * data['total_hours_watched'] + 
         0.1 * (data['payment_method'] == 'Mailed Check'))

data['churn_prob'] = 1 / (1 + np.exp(-logit))
data['churned'] = (data['churn_prob'] > np.random.uniform(0.4, 0.6, n_subs)).astype(int)

print(data['churned'].value_counts())

# 2. Saving the Dataset
data.to_csv('data/saas_churn_data.csv', index=False)

NameError: name 'os' is not defined

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# 1. Prepping Data (Converting 'payment_method' to dummy variables)
X = pd.get_dummies(data.drop(['user_id', 'churn_prob', 'churned'], axis=1), drop_first=True)
y = data['churned']

# 2. Splitting Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Training Model
# We use class_weight='balanced' because our churners are a minority
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# 4. Evaluation
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

NameError: name 'pd' is not defined

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh', color='teal')
plt.title('Top Drivers of Customer Churn')
plt.savefig('visuals/Top_Drivers_of_Customer_Churn.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Set professional theme
sns.set_theme(style="white")

# 1. Feature Importance Bar Chart
# Extracting the importance from our trained 'model'
plt.figure(figsize=(10, 6))
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

sns.barplot(
    x=importances.values, 
    y=importances.index, 
    hue=importances.index, # Fix: Mapping hue to y-axis categories
    palette='magma', 
    legend=False           # Fix: Removing redundant legend
)
plt.title('Key Drivers of Customer Churn', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score (Information Gain)')
plt.ylabel('User Attribute')
plt.savefig('visuals/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Confusion Matrix Heatmap
# This shows how many churners we actually caught
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=['Stayed', 'Churned'], 
    yticklabels=['Stayed', 'Churned']
)
plt.title('Model Performance: Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.savefig('visuals/churn_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()